# Tutorial about Fourier Ring Correlation (FRC) analysis

Fourier Ring Correlation provides a resolution measure that takes both localization precision and label density into account.

In [ ]:
import logging
import warnings

from pathlib import Path
from copy import copy, deepcopy

%matplotlib inline

import numpy as np
import matplotlib.pyplot as plt

import locan as lc

In [ ]:
lc.show_versions(system=False, dependencies=False, verbose=False)

## FRC for dSTORM data

First, let us look at FRC for dSTORM data showing nuclear pore complexes.

In [ ]:
locdata = lc.load_npc()
print(f"Metadata:\n")
locdata.print_summary()

In [ ]:
lc.render_2d(locdata, rescale=lc.Trafo.EQUALIZE, bin_size=10);

In [ ]:
frc = lc.FourierRingCorrelation(bin_size=10, smooth=1)
frc.compute(locdata)
frc.plot();

The results are accessible in a dedicated `FrcResults` class that also provides a method to compute a threshold frequency or resolution (defined as the inverse threshold frequency).

In [ ]:
print(frc.results)
print(f"Threshold frequency:", round(frc.results.threshold_frequency(), 4))
print(f"Resolution:", round(frc.results.resolution()))

The two image representations used for FRC computation are provided as `Image` object. Note that the images are not intensity equalized as often done for visualization.

In [ ]:
frc.results.image.shape, frc.results.other_image.shape

In [ ]:
fig, axes = plt.subplots(nrows=1, ncols=2, figsize=(10, 7))
axes[0].imshow(frc.results.image.data, vmax=0.1)
axes[1].imshow(frc.results.other_image.data, vmax=0.1);

To reduce boundary effects a windowing function must be applied using scipy window functions. Often, a Tukey window with alpha 1/8 is used in SMLM.

In [ ]:
frc = lc.FourierRingCorrelation(bin_size=10, window=("tukey", 0.25), smooth=1)
frc.compute(locdata)
frc.plot();

In [ ]:
fig, axes = plt.subplots(nrows=1, ncols=2, figsize=(10, 7))
axes[0].imshow(frc.results.image.data, vmax=0.1)
axes[1].imshow(frc.results.other_image.data, vmax=0.1);

## FRC for dSTORM data on smaller regions of interest

Analyzing a smaller region of interest further indicates that using a window filter, typically a Tukey filter with alpha=0.8, makes the FRC result more reliable (meaning resolution becomes larger).

In [ ]:
locdata_roi = lc.select_by_condition(
    locdata, condition="position_x<8_000 and position_y<8_000"
)
locdata_roi.reduce()

print(f"Metadata:\n")
locdata_roi.print_summary()

In [ ]:
lc.render_2d(locdata_roi, bin_size=10, rescale=lc.Trafo.EQUALIZE);

In [ ]:
lc.render_2d(locdata_roi, bin_size=10, rescale=lc.Trafo.STANDARDIZE, vmax=0.1);

In [ ]:
frc = lc.FourierRingCorrelation(bin_size=10, smooth=1)
frc.compute(locdata_roi)
frc.plot();

High frequency contributions result from boundary effects that should be reduced by applying window functions.

### Applying a Tukey window

In [ ]:
frc = lc.FourierRingCorrelation(bin_size=10, window=("tukey", 0.25), smooth=1)
frc.compute(locdata_roi)
frc.plot();

### Choosing the alpha parameter in Tukey windowing

When using a Tukey window, the precise value of alpha is not that important.

In [ ]:
frc_batch = []
windows = [None, ("tukey", 0)] + [("tukey", alpha_) for alpha_ in np.logspace(-2, 0, 10)]
for window_ in windows:
    frc = lc.FourierRingCorrelation(bin_size=10, window=window_, smooth=1)
    frc.compute(locdata_roi)
    frc_batch.append(frc)

In [ ]:
frc_batch

In [ ]:
for frc_ in frc_batch:
    frc_.plot()

In [ ]:
fig, axes = plt.subplots(nrows=1, ncols=5, figsize=(15, 5))
axes[0].imshow(frc_batch[0].results.image.data, cmap="cet_fire", vmax=1)
axes[1].imshow(frc_batch[1].results.image.data, cmap="cet_fire", vmax=1)
axes[2].imshow(frc_batch[2].results.image.data, cmap="cet_fire", vmax=1)
axes[3].imshow(frc_batch[7].results.image.data, cmap="cet_fire", vmax=1)
axes[4].imshow(frc_batch[-1].results.image.data, cmap="cet_fire", vmax=1);

## Detailed FRC workflow

For further choices with regard to image preparation make use of upstream analysis functions. The following examples show the influence of histogram equalization and windowing on the FRC.

In [ ]:
collection = lc.LocData.from_chunks(locdata=locdata_roi, n_chunks=2, order="alternating")
bin_range = lc.range_from_collection(locdatas=collection.references)
bins = lc.Bins(bin_range=bin_range, bin_size=10)
images = [
    lc.histogram(locdata=locdata_, bins=bins).data for locdata_ in collection.references
]

In [ ]:
images_standardized = [
    lc.adjust_contrast(image_, rescale=lc.Trafo.STANDARDIZE) for image_ in images
]
images_equalized = [
    lc.adjust_contrast(image_, rescale=lc.Trafo.EQUALIZE) for image_ in images
]
images_windowed = [
    lc.apply_window(image_, window_function="tukey", alpha=0.25) for image_ in images
]
images_equalized_windowed = [
    lc.apply_window(image_, window_function="tukey", alpha=0.25)
    for image_ in images_equalized
]

In [ ]:
fig, axes = plt.subplots(nrows=1, ncols=3, figsize=(15, 5))
axes[0].imshow(images[0], cmap="cet_fire")
axes[1].imshow(images_equalized[0], cmap="cet_fire")
axes[2].imshow(images_equalized_windowed[0], cmap="cet_fire");

In [ ]:
for label_, imgs_ in zip(
    ["original", "standardized", "equalized", "windowed", "equalized_windowed"],
    [
    images,
    images_standardized,
    images_equalized,
    images_windowed,
    images_equalized_windowed,
]):
    img = imgs_[0]
    print(
        label_  + ": ",
        f"Min intensity: {img.min():0.2f}",
        f"Max intensity: {img.max():0.2f}",
        "Data type:", img.dtype,
    )

Compute the FRC for all image representations.

In [ ]:
images_preprocessed = lc.analysis.fourier_ring_correlation.preprocess_images(*images)
frc_result = lc.analysis.fourier_ring_correlation.compute_fourier_ring_correlation(
    *images_preprocessed, bin_size=10
)

In [ ]:
images_preprocessed = lc.analysis.fourier_ring_correlation.preprocess_images(
    *images_standardized
)
frc_result_standardized = (
    lc.analysis.fourier_ring_correlation.compute_fourier_ring_correlation(
        *images_preprocessed, bin_size=10
    )
)

In [ ]:
images_preprocessed = lc.analysis.fourier_ring_correlation.preprocess_images(
    *images_equalized
)
frc_result_equalized = (
    lc.analysis.fourier_ring_correlation.compute_fourier_ring_correlation(
        *images_preprocessed, bin_size=10
    )
)

In [ ]:
images_preprocessed = lc.analysis.fourier_ring_correlation.preprocess_images(
    *images_windowed
)
frc_result_windowed = (
    lc.analysis.fourier_ring_correlation.compute_fourier_ring_correlation(
        *images_preprocessed, bin_size=10
    )
)

In [ ]:
images_preprocessed = lc.analysis.fourier_ring_correlation.preprocess_images(
    *images_equalized_windowed
)
frc_result_equalized_windowed = (
    lc.analysis.fourier_ring_correlation.compute_fourier_ring_correlation(
        *images_preprocessed, bin_size=10
    )
)

Plot the results

In [ ]:
plt.plot(frc_result.frequency, frc_result.correlation, label="original")
plt.plot(
    frc_result_standardized.frequency,
    frc_result_standardized.correlation,
    label="standardized",
)
plt.plot(
    frc_result_equalized.frequency, frc_result_equalized.correlation, label="equalized"
)
plt.plot(
    frc_result_windowed.frequency, frc_result_windowed.correlation, label="windowed"
)
plt.plot(
    frc_result_equalized_windowed.frequency,
    frc_result_equalized_windowed.correlation,
    label="equalized and windowed",
)
plt.legend();

## Linking localizations

The fact, that in dSTORM multiple localizations arise from the same emitter, results in an apparent resolution improvement due those repeated detection events varying in space by just the localization precision.

Linking localizations that appear in successive frames and that are close in space, reduces this spurious resolution contribution.

In [ ]:
locdata_smaller_roi = lc.select_by_condition(
    locdata, condition="6_000 < position_x < 7_000 and 4_000 < position_y < 5_000"
)
locdata_smaller_roi.reduce()

print(f"Metadata:\n")
locdata_smaller_roi.print_summary()

In [ ]:
lc.render_2d(locdata_smaller_roi, bin_size=10, rescale=lc.Trafo.EQUALIZE);

In [ ]:
frc = lc.FourierRingCorrelation(bin_size=10, window=("tukey", 0.25), smooth=1)
frc.compute(locdata_smaller_roi)
frc.plot();

In [ ]:
trackpy_logger = logging.getLogger("trackpy")
trackpy_logger.setLevel(logging.WARN)

In [ ]:
%%time
locdata_linked, _ = lc.track(locdata_smaller_roi, search_range=50, memory=10)
locdata_linked.region = locdata.region
locdata_linked.reduce(reset_index=True)
locdata_linked.print_summary()

In [ ]:
lc.render_2d(locdata_linked, bin_size=10, rescale=lc.Trafo.EQUALIZE);

In [ ]:
frc = lc.FourierRingCorrelation(bin_size=10, window=("tukey", 0.25), smooth=1)
frc.compute(locdata_linked)
frc.plot();